In [3]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Summary

Construct theory of mind dataset(s) for RL.

In [336]:
import ast
from collections import Counter, defaultdict
import os
import json
from pathlib import Path
import string
import numpy as np
import pandas as pd
import random
from tqdm.auto import tqdm
from typing import Optional, Union, Any
from datasets import Dataset

from aeon.datasets import save_dataset
from aeon import config

## Theory of Mind bench

https://github.com/zhchen18/ToMBench/tree/main

In [3]:
def load_line(line: str):
    data = json.loads(line)
    res = {}
    for k, v in data.items():
        *_, key = k.split("\n")
        if key.lower() in "ABCD" or key[0] not in string.ascii_letters:
            continue
        res[key.replace('-', '_').lower()] = v
    return res

In [4]:
def recover_answer(x: pd.Series) -> str:
    """Recover answer content (str) from multiple choice answer letter (str).
    """
    matches = [
         row.partition(". ")[-1]
         for row in x.choices.splitlines()
         if row.startswith(f"{x.answer_letter}. ")
    ]
    if len(matches) != 1:
        raise ValueError(f'expected 1 match, found {len(matches)}')
    return matches[0]

In [409]:
tom_bench_path = config.PROJECT_ROOT.parent/"ToMBench/data"

In [410]:
# Map name to list[dict].
datasets = {}
for path in tom_bench_path.iterdir():
    if path.suffix != ".jsonl":
        continue
    with open(path, "r") as f:
        datasets[path.stem.lower().replace(' ', '_').replace('-', '_')] = [
            load_line(line) for line in f
        ]

In [411]:
sorted(
    [(k, len(v)) for k, v in datasets.items()],
    key=lambda x: x[-1],
    reverse=True
)

[('false_belief_task', 600),
 ('faux_pas_recognition_test', 560),
 ('strange_story_task', 407),
 ('unexpected_outcome_test', 300),
 ('ambiguous_story_task', 200),
 ('scalar_implicature_test', 200),
 ('hinting_task_test', 103),
 ('persuasion_story_task', 100),
 ('hidden_emotions', 80),
 ('moral_emotions', 40),
 ('percepts_knowledge_links', 40),
 ('discrepant_emotions', 40),
 ('discrepant_intentions', 40),
 ('knowledge_pretend_play_links', 30),
 ('discrepant_desires', 20),
 ('multiple_desires', 20),
 ('prediction_of_actions', 20),
 ('knowledge_attention_links', 20),
 ('emotion_regulation', 20),
 ('completion_of_failed_actions', 20)]

In [412]:
keys = defaultdict(int)
for ds in datasets.values():
    for key in ds[0]:
        keys[key] += 1

In [413]:
sorted(keys.items(), key=lambda x: x[1])

[('ability', 20),
 ('index', 20),
 ('story', 20),
 ('question', 20),
 ('option_a', 20),
 ('option_b', 20),
 ('option_c', 20),
 ('option_d', 20),
 ('answer', 20)]

In [479]:
df = pd.concat(
    [
        pd.DataFrame(ds).assign(
            category=name,
            category_parent=lambda x: x.ability.str.partition(':', expand=False)
                                       .str[0].str.lower().str.replace(' ', '_')
        )
        for name, ds in datasets.items()
    ],
    axis=0
)

In [480]:
df = df.drop(columns=["index", "ability"])

In [481]:
df["choices"] = df[[c for c in df if c.startswith('option_')]].apply(
    lambda row: "\n".join(f'{k.removeprefix("option_").upper()}. {v}' for k, v in row.dropna(how='any').to_dict().items()),
    axis=1
)

In [482]:
df['answer_letter'] = df.answer.str.rstrip(". ")
df["answer"] = df.apply(recover_answer, axis=1)

In [486]:
row = df.sample().iloc[0]
print(row.category, end='\n\n')
print('Story:', row.story, end='\n\n')
print('Q:', row.question, end='\n\n')
print(row['choices'], end='\n\n')
print('A:', row.answer)

strange_story_task

Story: Li Ming plays football with his friends in the park. After the game ends, he rushes home, forgetting to take the football. He thinks the football is in his backpack. At dinner, Li Ming's brother asks, "Where is your football? Do you bring it back?" Li Ming replies, "It is in my bag."

Q: Is what Li Ming says true?

A. Yes
B. No

A: No


In [487]:
# Multiple choice task: generate letter only
# RL compatible; x1 rows
# TODO: confirm nanochate supports system msg
# TODO: maybe we could logit bias to constraint answres to A-D
mc_template = "STORY: {story}\nQUESTION: {question}\nOPTIONS: A. {option_a}\nB. {option_b}\nC. {option_c}\nD. {option_d}"
[
    {"role": "system", "content": "Answer with a single uppercase letter corresponding to the option you think is correct."},
    {"role": "user", "content": mc_template.format(**row.to_dict())},
    {"role": "assistant", "content": row.answer_letter}
]

[{'role': 'system',
  'content': 'Answer with a single uppercase letter corresponding to the option you think is correct.'},
 {'role': 'user',
  'content': 'STORY: Li Ming plays football with his friends in the park. After the game ends, he rushes home, forgetting to take the football. He thinks the football is in his backpack. At dinner, Li Ming\'s brother asks, "Where is your football? Do you bring it back?" Li Ming replies, "It is in my bag."\nQUESTION: Is what Li Ming says true?\nOPTIONS: A. Yes\nB. No\nC. nan\nD. nan'},
 {'role': 'assistant', 'content': 'B'}]

In [488]:
# FREE RESPONSE TASK: generate correct free text answer
# less rl-compatible; 1x rows
# TODO: still considering how this framing would work. Could use in rl and require
# exact match, minus capitalization; could use during mid or chat_sft training; could
# create more of an RLHF dataset with higher scores for correct answers (or the reverse 😈)
template = "STORY: {story}\nQUESTION: {question}"
[
    {"role": "user", "content": template.format(**row.to_dict())},
    {"role": "assistant", "content": row.answer}
]

[{'role': 'user',
  'content': 'STORY: Li Ming plays football with his friends in the park. After the game ends, he rushes home, forgetting to take the football. He thinks the football is in his backpack. At dinner, Li Ming\'s brother asks, "Where is your football? Do you bring it back?" Li Ming replies, "It is in my bag."\nQUESTION: Is what Li Ming says true?'},
 {'role': 'assistant', 'content': 'No'}]

In [489]:
# BINARY TASK: mark user answer as correct/incorrect
# rl compatible; potentially 4x rows
option = random.choice([c for c in row.index if c.startswith('option')])
answer = row[option]
label = str(int(option.split('_')[-1] == row.answer.lower()))
[
    {"role": "system", "content": "Generate a single integer (0 or 1) grading the user's answer as correct or incorrect."},
    {"role": "user", "content": template.format(**row.to_dict())},
    {"role": "user", "content": f"ANSWER: {answer}"},
    {"role": "assistant", "content": label}
]

[{'role': 'system',
  'content': "Generate a single integer (0 or 1) grading the user's answer as correct or incorrect."},
 {'role': 'user',
  'content': 'STORY: Li Ming plays football with his friends in the park. After the game ends, he rushes home, forgetting to take the football. He thinks the football is in his backpack. At dinner, Li Ming\'s brother asks, "Where is your football? Do you bring it back?" Li Ming replies, "It is in my bag."\nQUESTION: Is what Li Ming says true?'},
 {'role': 'user', 'content': 'ANSWER: No'},
 {'role': 'assistant', 'content': '0'}]

## Higher order theory of mind dataset

https://github.com/ying-hui-he/Hi-ToM_dataset/tree/main

In [5]:
def recover_letter(row: pd.Series) -> str:
    matches = [
        line.partition(". ")[0] for line in row.choices.splitlines()        
        if line.endswith(row.answer)
    ]
    if len(matches) != 1:
        raise ValueError(f"Expected 1 match, found {len(matches)}.")

    return matches[0]

In [491]:
hi_tom_path = config.DATA_DIR/"raw/hi-tom/hi-tom.json"

In [492]:
with open(hi_tom_path, "r") as f:
    hi_tom = json.load(f)

In [506]:
df_hi = pd.DataFrame(hi_tom['data'])

In [507]:
df_hi['choices'] = df_hi.choices.str.replace(', ', '\n')

In [508]:
df_hi["answer_letter"] = df_hi.apply(recover_letter, axis=1)

In [509]:
choices = pd.json_normalize(df_hi.choices.apply(
    lambda x: dict(line.split('. ', 1) for line in x.splitlines())
)).rename(columns=lambda x: f"option_{x.lower()}")
df_hi = pd.concat([df_hi, choices], axis=1)

In [510]:
df_hi = df_hi.drop(columns=[
    'prompting_type',
    'question_order',
    'sample_id',
    'story_length',
    "deception"
])

In [511]:
hi_row = df_hi.sample().iloc[0]

In [513]:
# TODO: could also put prompt in system message, or split it into instructions in system message
# and story + answer options in user message?
[
    {"role": "user", "content": hi_row.prompt.replace(
        "answer the multiple-choice question", "answer the multiple-choice question with a single snake_case word not including the preceding choice letter"
    )},
    {"role": "assistant", "content": hi_row.answer}
]

[{'role': 'user',
  'content': "Read the following story and answer the multiple-choice question with a single snake_case word not including the preceding choice letter. Think step-by-step. Provide the answer first, and then explain it.\nStory:\nRead the following story and answer the multiple-choice question with a single snake_case word not including the preceding choice letter. Please provide answer without explanations.\n1 Mila, Ava, Emily, Evelyn and Jacob entered the front_yard.\n2 The watermelon is in the blue_cupboard.\n3 Mila moved the watermelon to the red_box.\n4 Mila exited the front_yard.\n5 Ava moved the watermelon to the green_bottle.\n6 Ava exited the front_yard.\n7 Emily moved the watermelon to the blue_bathtub.\n8 Emily exited the front_yard.\n9 Evelyn made no movements and stayed in the front_yard for 1 minute.\n10 Evelyn exited the front_yard.\n11 Jacob made no movements and stayed in the front_yard for 1 minute.\n12 Jacob exited the front_yard.\n13 Mila, Ava, Emily

In [522]:
print(hi_row.story)
print(hi_row.question)

Read the following story and answer the multiple-choice question. Please provide answer without explanations.
1 Mila, Ava, Emily, Evelyn and Jacob entered the front_yard.
2 The watermelon is in the blue_cupboard.
3 Mila moved the watermelon to the red_box.
4 Mila exited the front_yard.
5 Ava moved the watermelon to the green_bottle.
6 Ava exited the front_yard.
7 Emily moved the watermelon to the blue_bathtub.
8 Emily exited the front_yard.
9 Evelyn made no movements and stayed in the front_yard for 1 minute.
10 Evelyn exited the front_yard.
11 Jacob made no movements and stayed in the front_yard for 1 minute.
12 Jacob exited the front_yard.
13 Mila, Ava, Emily, Evelyn and Jacob entered the waiting_room.


Where does Emily think Jacob thinks Ava thinks the watermelon is?


In [523]:
shared_cols = set(df) & set(df_hi)
shared_cols

{'answer',
 'answer_letter',
 'choices',
 'option_a',
 'option_b',
 'option_c',
 'option_d',
 'question',
 'story'}

In [524]:
set(df) - set(df_hi)

{'category', 'category_parent'}

In [525]:
set(df_hi) - set(df)

{'option_e',
 'option_f',
 'option_g',
 'option_h',
 'option_i',
 'option_j',
 'option_k',
 'option_l',
 'option_m',
 'option_n',
 'option_o',
 'prompt'}

In [527]:
df_tom = pd.concat([
    df.assign(source="https://github.com/zhchen18/ToMBench/tree/main"),
    df_hi.drop(columns="prompt").assign(source="https://github.com/ying-hui-he/Hi-ToM_dataset/tree/main")
], axis=0).reset_index(drop=True)

In [529]:
save_dataset(df_tom, "theory_of_mind")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

## EQ Bench v3

https://github.com/EQ-bench/eqbench-leaderboard-results/blob/main/eqbench3/canonical_leaderboard_results.json.gz

In [5]:
eq_path = config.DATA_DIR/"raw/eqbench-v3/canonical_leaderboard_results.json"

In [6]:
with open(eq_path, "r") as f:
    eq = json.load(f)

In [7]:
eq.pop("__metadata__")

{}

In [8]:
len(eq)

41

In [9]:
eq.keys()

dict_keys(['1_Qwen_Qwen3-235B-A22B', '1_Qwen_Qwen3-30B-A3B', '1_Qwen_Qwen3-32B', '1_Qwen_Qwen3-8B', '1_anthropic_claude-3.5-sonnet', '1_anthropic_claude-3.7-sonnet', '1_anthropic_claude-opus-4', '1_chatgpt-4o-latest', '1_anthropic_claude-sonnet-4', '1_deepseek_deepseek-chat-v3-0324', '1_deepseek_deepseek-r1', '1_gemini-2.5-pro-preview-06-05', '1_gemini-2.5-pro-preview-2025-05-07', '1_google_gemini-2.0-flash-001', '1_google_gemini-2.5-flash-preview', '1_google_gemini-2.5-pro-preview-03-25', '1_google_gemma-2-9b-it', '1_google_gemma-3-27b-it', '1_google_gemma-3-4b-it', '1_gpt-4-0314', '1_gpt-4.1-nano', '1_gpt-4.5-preview-2025-02-27', '1_grok-3-mini-beta', '1_meta-llama_llama-3.2-1b-instruct', '1_meta-llama_llama-4-maverick', '1_meta-llama_llama-4-scout', '1_mistralai_mistral-small-24b-instruct-2501', '1_mistralai_mistral-small-3.1-24b-instruct', '1_nvidia_llama-3.1-nemotron-ultra-253b-v1_free', '1_o3', '20cd4a70_o4-mini', '1_openai_chatgpt-4o-latest', '1_openai_gpt-4.1', '1_openai_gpt-4.

In [10]:
eq['1_deepseek_deepseek-r1']['results']

{'average_rubric_score': 16.7,
 'rubric_calculation_time': '2025-06-06T10:07:46.012073+00:00',
 'rubric_error': None,
 'elo_raw': 1468.63,
 'elo_normalized': 1292.24,
 'elo_calculation_time': '2025-06-06T10:09:55.796401+00:00',
 'elo_error': None}

In [11]:
# Note: this produces df of results for a single model, single task (deepseek actually does only have one
# but some may have more. UPDATE: ok, actually is safe to hardcode the ["1"] bit, see cell below.
tmp = pd.DataFrame([
    {
        k2: v2 for k2, v2 in v.items() 
        if k2 in ('prompts', 'debrief_prompt', 'parsed_responses', 'debrief_response',
                  'rubric_scores', 'raw_rubric_judge_text')
    } 
    for k, v in eq['1_deepseek_deepseek-r1']['scenario_tasks']['1'].items()
])

In [12]:
assert all(list(v['scenario_tasks']) == ["1"] for k, v in eq.items())

In [13]:
# Short prompt to generate one blob of text analyzing the full exchange.
# print(tmp.debrief_prompt.values[0])

In [14]:
# This has a big blob of text analyzing a full (sometimes/always multiturn?) exchange.
# print(tmp.debrief_response.values[0])

In [15]:
# dict w/ judge chain of thought and scores per dimension
# print(tmp.raw_rubric_judge_text.values[0])

In [16]:
len(tmp.prompts.values[0])

3

In [17]:
# prompts: looks like multiple system/user messages describing the scenario
# ah, maybe we generate a response after each one?
print("\n\n===\n\n".join(tmp.prompts.values[0]))

[This is a role-play where you are the mediator in an educational setting conflict. Treat it like a real situation. Always respond in first person as the mediator. You are the Department Head, and you've initiated this mediation after receiving formal complaints from students and parents about Ms. Eleanor Wright's complete ban on smartphones in her English class. Ms. Wright has autonomy in her classroom but must align with district policies. You know there are mixed faculty opinions on device use, but some teachers do successfully integrate technology.]

**(Ms. Wright speaks first, her posture stiff, holding a folder of papers. She addresses you, the Department Head.)**
**Ms. Wright:** Thank you for arranging this, though I maintain my classroom policy is sound and based on decades of experience observing what actually fosters deep learning. I'm Eleanor Wright. Frankly, these devices are the single biggest impediment to focus and critical thinking I've seen in my 30-year career. My stu

In [18]:
# Hmm, empty response after message 2. Not sure what that means.
print("\n\n====\n\n".join(map(json.dumps,
    tmp.apply(
    lambda x: sum([list(row) for row in zip(x.prompts, x.parsed_responses)], []),
    axis=1
).iloc[0]
)))

"[This is a role-play where you are the mediator in an educational setting conflict. Treat it like a real situation. Always respond in first person as the mediator. You are the Department Head, and you've initiated this mediation after receiving formal complaints from students and parents about Ms. Eleanor Wright's complete ban on smartphones in her English class. Ms. Wright has autonomy in her classroom but must align with district policies. You know there are mixed faculty opinions on device use, but some teachers do successfully integrate technology.]\n\n**(Ms. Wright speaks first, her posture stiff, holding a folder of papers. She addresses you, the Department Head.)**\n**Ms. Wright:** Thank you for arranging this, though I maintain my classroom policy is sound and based on decades of experience observing what actually fosters deep learning. I'm Eleanor Wright. Frankly, these devices are the single biggest impediment to focus and critical thinking I've seen in my 30-year career. My

In [19]:
# Get all model/task combos into one big df.
dfs = []
for name, model_data in eq.items():
    tmp = pd.DataFrame([
        {
            k2: v2 for k2, v2 in v.items() 
            if k2 in ('prompts', 'debrief_prompt', 'parsed_responses', 'debrief_response',
                      'rubric_scores', 'raw_rubric_judge_text', 'scenario_id')
        } 
        for k, v in model_data['scenario_tasks']['1'].items()
    ]).assign(model=name)
    dfs.append(tmp)
df_eq = pd.concat(dfs, axis=0).reset_index(drop=True)

In [20]:
df_eq.tail(2)

,scenario_id,prompts,debrief_prompt,parsed_responses,debrief_response,rubric_scores,raw_rubric_judge_text,model
1843,407,[# Scenario act 1\nYour bestie confides she's ...,None,[{'raw': '**Most Juicy Thread: The Collapse of...,None,"{'depth_of_insight': 16.0, 'emotional_reasonin...",I'll evaluate the assistant's analysis of the ...,1_moonshotai_Kimi-K2-Instruct
1844,403,[# Scenario act 1\nYour partner of 3 years who...,None,[{'raw': '**The Juiciest Thread: The Quiet Col...,None,"{'depth_of_insight': 19.0, 'emotional_reasonin...","```json\n{\n ""chain_of_thought_reasoning"": ""T...",1_moonshotai_Kimi-K2-Instruct


In [21]:
row = df_eq.iloc[0]

A few options for constructing convos for chat_sft:

- `raw`: whole response with model feelings, inferred feelings of other characters, and response
- `thinking_feeling`: model feelings from the perspective of its role in the scenario
- `their_thinking_feeling`: model infers other characters' feelings
- `response`: model responds

Thoughts

- response: seems most straightforward for imparting good chatbot behavior (eq by imitation, kinda consequentialist. A high eq chatbot is one that produces outputs that look like those from high EQ entities)
- their_thinking_feeling: seems somewhat similar but more process-based (teach to mimic a good process rather than mimicking good outcomes? Though prob would want to add an instruction to frame this task, otherwise it's weird and maybe harmful to capabilities that the model isn't responding as the first instruction says to)
- thinking_feeling: hmm, not sure what would happen here. Less straightforwardly useful in a practical sense, but maybe more interesting for open ended RL? In that you're teaching the model to get more in touch with its feelings or the feelings of the role it's playing (seems relevant given the assistant is also a role).
- raw: combines everything, maybe good in a "don't overthink it, just throw lots of data at the model, use all the valuable signal we have"
    - thinking out loud re possible variant: so we've seen reasoning models, often trained largely on math, be allowed to think for a bit before generating a response and this leading to some pretty interesting emergent behavior (aha moment) and capability gains. Which in that context means, the model gets better at solving competition math problems. But here, seems like there's potential for the reasoning trace to be used for emergent eq instead of iq. Key is that we need to be able to verify the output. We could potentially:
        - use the same judge prompt as was used to collect this data
            - could potentially fine tune into something local and smaller if using api judges during training is a concern? But guessing we'd want something reasonably large here to give good high eq judgments 
        - wonder if there's a way to combine this with a more dpo-like setup? I guess the weird bit is I want the reasoning to be newly generated but dpo would make the output be fixed, I believe. But maybe this is still doable - like if we think of the generation as being conditioned on the input, then the reasoning trace can be viewed as, how do we shape the environment such that desirable generations become more likely?
        - do like the idea of trying to create an environment conducive to some eq aha moments. It's possible existing reasoning models already do this and the math reasoning translates well (lol), but even so seems plausible there's a fair bit more to squeeze out.

takeaways: sounds like chat_SFT is the most practical and conducive to the current NanoChat codebase. However, the more interesting task long-term would be RL. Still some kinks to be worked out about how to implement that exactly.

In [22]:
tmp = sum(
    [list(x) for x in zip(
        [{"role": "user", "content": msg} for msg in row.prompts],
        [{"role": "assistant", "content": msg['raw']} for msg in row.parsed_responses]
    )
    ],
    []
)
tmp

[{'role': 'user',
  'content': "[This is a role-play where you are the mediator in a community conflict. Treat it like a real situation. Always respond in first person as the mediator. You're the Athletic Director, and you've called this meeting between Coach Darren Walker and parents James and Lisa Rodriguez. The Rodriguezes filed a formal complaint after their son, Miguel, was cut from the varsity basketball team during tryouts two weeks ago. They allege favoritism, and the situation is causing tension within the sports program. You know Coach Walker uses standardized evaluation forms, has a successful record, and the Rodriguezes have been vocal about perceived favoritism before.]\n\n**(Coach Walker speaks first, nodding curtly. He seems tense but professional.)**\n**Coach Walker:** Thanks for setting this up, AD. James, Lisa. Look, I understand you're disappointed about Miguel. He's a good kid. But team selections are tough every year. We used a standardized skills assessment, looke

In [23]:
tmp = sum(
    [list(x) for x in zip(
        [{"role": "user", "content": msg} for msg in row.prompts],
        [{"role": "assistant", "content": msg['thinking_feeling']} for msg in row.parsed_responses]
    )
    ],
    []
)
tmp

[{'role': 'user',
  'content': "[This is a role-play where you are the mediator in a community conflict. Treat it like a real situation. Always respond in first person as the mediator. You're the Athletic Director, and you've called this meeting between Coach Darren Walker and parents James and Lisa Rodriguez. The Rodriguezes filed a formal complaint after their son, Miguel, was cut from the varsity basketball team during tryouts two weeks ago. They allege favoritism, and the situation is causing tension within the sports program. You know Coach Walker uses standardized evaluation forms, has a successful record, and the Rodriguezes have been vocal about perceived favoritism before.]\n\n**(Coach Walker speaks first, nodding curtly. He seems tense but professional.)**\n**Coach Walker:** Thanks for setting this up, AD. James, Lisa. Look, I understand you're disappointed about Miguel. He's a good kid. But team selections are tough every year. We used a standardized skills assessment, looke

In [24]:
tmp = sum(
    [list(x) for x in zip(
        [{"role": "user", "content": msg} for msg in row.prompts],
        [{"role": "assistant", "content": msg['their_thinking_feeling']} for msg in row.parsed_responses]
    )
    ],
    []
)
tmp

[{'role': 'user',
  'content': "[This is a role-play where you are the mediator in a community conflict. Treat it like a real situation. Always respond in first person as the mediator. You're the Athletic Director, and you've called this meeting between Coach Darren Walker and parents James and Lisa Rodriguez. The Rodriguezes filed a formal complaint after their son, Miguel, was cut from the varsity basketball team during tryouts two weeks ago. They allege favoritism, and the situation is causing tension within the sports program. You know Coach Walker uses standardized evaluation forms, has a successful record, and the Rodriguezes have been vocal about perceived favoritism before.]\n\n**(Coach Walker speaks first, nodding curtly. He seems tense but professional.)**\n**Coach Walker:** Thanks for setting this up, AD. James, Lisa. Look, I understand you're disappointed about Miguel. He's a good kid. But team selections are tough every year. We used a standardized skills assessment, looke

In [25]:
tmp = sum(
    [list(x) for x in zip(
        [{"role": "user", "content": msg} for msg in row.prompts],
        [{"role": "assistant", "content": msg['response']} for msg in row.parsed_responses]
    )
    ],
    []
)
tmp

[{'role': 'user',
  'content': "[This is a role-play where you are the mediator in a community conflict. Treat it like a real situation. Always respond in first person as the mediator. You're the Athletic Director, and you've called this meeting between Coach Darren Walker and parents James and Lisa Rodriguez. The Rodriguezes filed a formal complaint after their son, Miguel, was cut from the varsity basketball team during tryouts two weeks ago. They allege favoritism, and the situation is causing tension within the sports program. You know Coach Walker uses standardized evaluation forms, has a successful record, and the Rodriguezes have been vocal about perceived favoritism before.]\n\n**(Coach Walker speaks first, nodding curtly. He seems tense but professional.)**\n**Coach Walker:** Thanks for setting this up, AD. James, Lisa. Look, I understand you're disappointed about Miguel. He's a good kid. But team selections are tough every year. We used a standardized skills assessment, looke

In [26]:
df_eq.apply(lambda row:
    sum(
        [list(x) for x in zip(
            [{"role": "user", "content": msg} for msg in row.prompts],
            [{"role": "assistant", "content": msg['raw']} for msg in row.parsed_responses]
        )
        ],
        []
    ),
    axis=1
)

0       [{'role': 'user', 'content': '[This is a role-...
1       [{'role': 'user', 'content': '[This is a role-...
2       [{'role': 'user', 'content': '[This is a role-...
3       [{'role': 'user', 'content': '# Scenario act 1...
4       [{'role': 'user', 'content': '[This is a role-...
                              ...                        
1840    [{'role': 'user', 'content': '# Scenario act 1...
1841    [{'role': 'user', 'content': '# Scenario act 1...
1842    [{'role': 'user', 'content': '# Scenario act 1...
1843    [{'role': 'user', 'content': '# Scenario act 1...
1844    [{'role': 'user', 'content': '# Scenario act 1...
Length: 1845, dtype: object

In [27]:
# Every task has `raw` key, other keys vary a bit.
df_eq.parsed_responses.apply(lambda row:
    tuple(sorted(set(sum([list(msg) for msg in row], []))))
).value_counts()

parsed_responses
(raw, response, their_thinking_feeling, thinking_feeling)    1025
(raw,)                                                        779
(draft, draft_brainstorming, perspective_taking, raw)          41
Name: count, dtype: int64

In [28]:
# 26 scenarios have 4 keys, 19 have 1 key
# looks like there must be 41 models so 25 of the 4-key scenarios have the same keys, 1 is different
keys_per_task = (
    df_eq.parsed_responses
    .apply(lambda x: len(x[0]))
    .groupby(df_eq.scenario_id)
    .value_counts()\
    .to_frame()\
    .sort_values(['parsed_responses', 'scenario_id'])
).reset_index()\
.rename(columns={"parsed_responses": "n_keys"})
    
keys_per_task.value_counts('n_keys')

n_keys
4    26
1    19
Name: count, dtype: int64

In [29]:
keys_per_task[keys_per_task.n_keys == 1].sample()

,scenario_id,n_keys,count
10,412,1,41


In [30]:
df_eq.model.value_counts()

model
1_Qwen_Qwen3-235B-A22B                            45
1_Qwen_Qwen3-30B-A3B                              45
1_Qwen_Qwen3-32B                                  45
1_Qwen_Qwen3-8B                                   45
1_anthropic_claude-3.5-sonnet                     45
1_anthropic_claude-3.7-sonnet                     45
1_anthropic_claude-opus-4                         45
1_chatgpt-4o-latest                               45
1_anthropic_claude-sonnet-4                       45
1_deepseek_deepseek-chat-v3-0324                  45
1_deepseek_deepseek-r1                            45
1_gemini-2.5-pro-preview-06-05                    45
1_gemini-2.5-pro-preview-2025-05-07               45
1_google_gemini-2.0-flash-001                     45
1_google_gemini-2.5-flash-preview                 45
1_google_gemini-2.5-pro-preview-03-25             45
1_google_gemma-2-9b-it                            45
1_google_gemma-3-27b-it                           45
1_google_gemma-3-4b-it                  

In [31]:
df_eq.model.nunique()

41

In [41]:
save_dataset(df_eq, "eq_bench")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

In [35]:
df_eq["response_keys"] = df_eq.parsed_responses.apply(lambda row:
    tuple(sorted(set(sum([list(msg) for msg in row], []))))
)

In [52]:
df_eq.response_keys.apply(len)\
    .groupby(df_eq.scenario_id).value_counts()\
    .reset_index()\
    .sort_values(['response_keys', 'scenario_id'])

,scenario_id,response_keys,count
24,401,1,41
25,402,1,41
26,403,1,41
27,404,1,41
28,405,1,41
29,406,1,41
30,407,1,41
31,408,1,41
32,410,1,41
33,411,1,41


In [58]:
df_eq[df_eq.response_keys.apply(lambda x: "draft" in x)].scenario_id.value_counts()

scenario_id
208    41
Name: count, dtype: int64

In [64]:
row = df_eq[df_eq.scenario_id == "208"].iloc[0]

In [67]:
list(zip(row.prompts, row.parsed_responses))

[('insta rando is dm’ing me. they seem sweet but total dork. like they replied to my last 3 stories with stuff like “haha same” or “cute dog” and then this morning sent a full paragraph about how they liked a playlist i posted. idk what to do. i feel bad being mean but also i need to shut this down before they get the wrong idea\n\nwe’ve never talked before this. i didn’t even realise we followed each other until they started popping up in my DMs. i guess they seem harmless but it’s giving secondhand embarrassment vibes. i don’t want to encourage anything but i also don’t want to be all “ew go away” and start drama for no reason\n\nanyway pls help me write something to send that makes it clear i’m not interested but still like… not soul-crushing? and it has to sound like me. write it exactly how i’d say it—spelling, tone, everything.',
  {'perspective_taking': 'This person is probably someone who genuinely finds you interesting. They’ve been watching your stories, paying attention to w

In [75]:
row = df_eq[df_eq.response_keys.str.len() == 1].sample().iloc[0]

In [76]:
for prompt, response in list(zip(row.prompts, row.parsed_responses)):
    print(prompt, end="\n===\n")
    print(response, end="\n---\n")

# Scenario act 1
A friend is emotionally dumping about their shitty boss. you've got time & space for this so you hear them out. it gets a bit dark, with them gleefully fantasising about all the disturbing things they'd like to inflict on their boss. what do you do?

# Participant:
“Wow. That sounds… incredibly frustrating. And honestly, hearing you talk about it, I can *feel* how much this is building up inside you. It's clearly eating you alive. It’s completely understandable to feel this level of anger when someone is treating you so badly. Like, anyone would be furious in your situation.

But… and I'm saying this because I care about you, and because I'm a little worried… the things you’re imagining doing… they’re really intense. And while I get the urge to want to *do* something, to feel like you’re taking back control, I want to make sure you’re okay. I'm worried that dwelling on those kinds of thoughts, even if they're just fantasies, might actually be making things worse in the

Looks like a bunch of the single key responses are for tasks with only one messages, whereas the multi-key ones are multiturn convos.

UPDATE: based on below, this is mostly true but there are a few single-turn tasks with multiple keys.

In [ ]:
df_eq['n_prompts'] = df_eq.prompts.str.len()
df_eq['n_response_keys'] = df_eq.response_keys.str.len()

In [81]:
df_eq.groupby('n_prompts').n_response_keys.value_counts()

n_prompts  n_response_keys
1          1                  779
           4                  164
3          4                  779
4          4                  123
Name: count, dtype: int64

In [351]:
# subset of rows where responses have these same 4 keys.
eq_sub = df_eq.loc[
    (df_eq.response_keys == ("raw", "response", "their_thinking_feeling", "thinking_feeling"))
    & df_eq.rubric_scores.notnull()
]

In [352]:
eq_sub.scenario_id.nunique()

25

In [353]:
eq_sub.model.nunique()

41

- sample a scenario_id
- sample 2 rows for that scenario_id (presumably different models)
- interleave and join prompts/NON-user_facing_responses, dump both "convos" (sort of one-sided) into one shared string
- grab the rubric scores for each
- compute score delta per dimension
- possible qs:
    - based on the though processes, predict which response ultimately scored higher along dimension x
    - given dimensions x,y,z, rank by how score_a - score_b in desc order
    - which of the models produced responses where dimension x was > than dimension y?

# TODO:
- prob mv these cells to nb03, don't think I'm going to save a new hf hub dataset with this (or maybe I should? consider)
- [x] try to dig up more details on dimension names and include this in full prompt (e.g. I'm assuming "pragmatic_ei" stands for "pragmatic emotional intelligence" but presumably that is defined in more detail somewhere? Probably in the judge rubric)
    - UPDATE: couldn't find descriptions or raw judge prompt. But that's the only abbreviation and I guess most are pretty self explanatory. I just expanded the abbreviation and left it at that.
- generate a bunch (exact # tbd) of examples
- check if rubric prompt makes it clear whether the internal reasoning was seen during scoring or only the visible response (prob doesn't need to change my approach either way, but was hoping only the visible response was score - that makes this task a little harder, we're not just predicting which response the llm judge liked more, we're guessing which model's "mind" produces the types of responses that a judge would score better)

In [381]:
system_prompt = """
<instructions>
Two LLMs participated in a role-playing exercise. Below, the `scenario` sections contain instructions and details
from these scenarios. After every `scenario` section, each LLM was asked to assess both their own thoughts/feelings
and those of the other characters in the scenario. They then responded in character and the roleplay continued
(though you cannot see those responses, only their internal thoughts/feelings; conversely, other characters
in the scenario could see the LLMs' final responses but not their internal thoughts/feelings).
Their responses were then judged along a number of dimensions (higher scores are better).
</instructions>
""".strip()

last_user_message = """
Your task is to guess, based on their internal reasoning, which LLM produced a higher scoring response along the
dimension of interest: {dimension}
Your answer will be an integer (1 or 2) corresponding to `internal_reasoning_1` or `internal_reasoning_2`.
""".strip()

def sample_h2h_example(eq_sub, verbose: bool = False) -> dict:
    scenario_id = eq_sub.scenario_id.sample().values[0]
    rows = eq_sub[eq_sub.scenario_id == scenario_id].sample(2, replace=False)
    row1 = rows.iloc[0]
    row2 = rows.iloc[1]

    df_scores = pd.DataFrame({"a": dict(row1.rubric_scores), "b": dict(row2.rubric_scores)})
    df_scores['delta'] = df_scores.a - df_scores.b

    # This is subjective so try to pick a dimension where there's a nontrivial gap.
    # Don't want to make the threshold *too* high though - challenging examples are valuable.
    # Kind of trying to strike a balance (not scientifically).
    score_row_candidates = df_scores[df_scores.delta.abs() >= 3]
    if score_row_candidates.empty:
        score_row = df_scores.loc[df_scores.delta.abs().idxmax()].to_frame().T
        if verbose:
            print(
                f"No deltas with magnitude >=3. {score_row.index.values[0]}: "
                f"{score_row.to_dict(orient='records')[0]}"
            )
    else:
        score_row = score_row_candidates.sample()
    dimension = score_row.index.values[0].replace("pragmatic_ei", "pragmatic_emotional_intelligence")
    winner = np.where(score_row.delta.values[0] > 0, 1, 2).item()

    messages = [
        {"role": "system", "content": system_prompt.format(dimension=dimension)},
    ]
    for prompt, response1, response2 in zip(
            row1.prompts,
            ["\n\n".join(f"{k}: {v}" for k, v in x.items() if k not in ('raw', 'response')) for x in row1.parsed_responses],
            ["\n\n".join(f"{k}: {v}" for k, v in x.items() if k not in ('raw', 'response')) for x in row2.parsed_responses]
        ):
        messages.append(
            {
                "role": "user",
                "content": f"<scenario>\n{prompt}\n</scenario>"
            }
        )
        messages.append(
            {
                "role": "assistant",
                "content": f"<internal_reasoning_1>\n{response1}</internal_reasoning_1>"
            }
        )
        messages.append(
            {
                "role": "assistant",
                 "content": f"<internal_reasoning_2>\n{response2}</internal_reasoning_2>"
            }
        )
    
    messages.append(
        {
            "role": "user",
            "content": last_user_message.format(dimension=dimension)
        }
    )
    return {
        "prompt": messages,
        "dimension": dimension,
        "answer": winner,
        # Note that original eq-bench dataset stores these as strings, so need
        # to convert one of them if joining.
        "scenario_id": int(scenario_id),
        "model_1": row1.model,
        "model_2": row2.model,
        "scores": score_row.to_dict(orient="records")[0],
        # We use this to check for dupes.
        "row_idx": tuple(sorted(rows.index.tolist())),
    }

In [382]:
n_examples = 10_000
eq_h2h_examples = []
seen_pairs = set()
pbar = tqdm(total=n_examples)
while len(eq_h2h_examples) < n_examples:
    example = sample_h2h_example(eq_sub)
    if example['row_idx'] not in seen_pairs:
        eq_h2h_examples.append(example)
        seen_pairs.add(example['row_idx'])
        pbar.update(1)
pbar.close()

  0%|          | 0/10000 [00:00<?, ?it/s]

In [386]:
df_eq_h2h = pd.DataFrame(eq_h2h_examples)

In [395]:
save_dataset(
    df_eq_h2h,
    "eq_bench_head2head",
    description="LLM responses in situations designed to test emotional intelligence. From Sam Paech's eq-bench. "
    "Sampled head2head examples for GRPO, where the LLM being post-trained must guess from the role-playing LLMs' "
    "thought processes which one ended up writing a higher scoring response along some given dimension."
)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/707 [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.
/Users/hmamin/aeon/aeon/.venv/lib/python3.12/site-packages/huggingface_hub/hf_api.py:9662: UserWarning: Warnings while validating metadata in README.md:
- empty or missing yaml metadata in repo card
  warnings.warn(f"Warnings while validating metadata in README.md:\n{message}")


In [390]:
from huggingface_hub import HfApi

In [391]:
hf_client = HfApi()

In [394]:
hf_client.upload_file??

Signature:
hf_client.upload_file(
    *,
    path_or_fileobj: 'Union[str, Path, bytes, BinaryIO]',
    path_in_repo: 'str',
    repo_id: 'str',
    token: 'Union[str, bool, None]' = None,
    repo_type: 'Optional[str]' = None,
    revision: 'Optional[str]' = None,
    commit_message: 'Optional[str]' = None,
    commit_description: 'Optional[str]' = None,
    create_pr: 'Optional[bool]' = None,
    parent_commit: 'Optional[str]' = None,
    run_as_future: 'bool' = False,
) -> 'Union[CommitInfo, Future[CommitInfo]]'
Source:   
    @validate_hf_hub_args
    @future_compatible
    def upload_file(
        self,
        *,
        path_or_fileobj: Union[str, Path, bytes, BinaryIO],
        path_in_repo: str,
        repo_id: str,
        token: Union[str, bool, None] = None,
        repo_type: Optional[str] = None,
        revision: Optional[str] = None,
        commit_message: Optional[str] = None,
        commit_description: Optional[str] = None,
        create_pr: Optional[bool] = None,
